# Animal alias conflict audit

Read-only audit of canonical animal names and aliases in `wildlife.db`. A conflict is a normalized name linked to more than one animal ID.

In [ ]:
from pathlib import Path
from collections import defaultdict
import re
import sqlite3
import unicodedata

DB = Path('../wildlife.db')

def normalized_name(value):
    value = unicodedata.normalize('NFC', value or '').strip().casefold()
    return ''.join(
        char for char in value
        if not char.isspace() and unicodedata.category(char) != 'Cf'
    )

connection = sqlite3.connect(DB)
connection.row_factory = sqlite3.Row
animals = {
    row['id']: dict(row)
    for row in connection.execute(
        'SELECT id, name_my, name_en, scientific_name FROM animals'
    )
}
links = []
for animal in animals.values():
    for field in ('name_my', 'name_en', 'scientific_name'):
        value = animal[field]
        if value:
            links.append((normalized_name(value), value, animal['id'], field))
for row in connection.execute('SELECT animal_id, alias_my FROM animal_aliases'):
    if row['alias_my']:
        links.append((normalized_name(row['alias_my']), row['alias_my'], row['animal_id'], 'alias'))
connection.close()

by_name = defaultdict(list)
for link in links:
    by_name[link[0]].append(link)
conflicts = {
    name: values for name, values in by_name.items()
    if len({value[2] for value in values}) > 1
}

print({
    'animal_records': len(animals),
    'name_links_checked': len(links),
    'conflict_names': len(conflicts),
    'affected_animals': len({value[2] for values in conflicts.values() for value in values}),
})
for name, values in sorted(conflicts.items()):
    ids = sorted({value[2] for value in values})
    print(name, ids, [(animal_id, animals[animal_id]['name_my'], animals[animal_id]['scientific_name']) for animal_id in ids])
